# పాఠం 12 - ఏజెంట్ స్క్రాచ్‌ప్యాడ్‌తో చాట్ చరిత్ర తగ్గింపు

దీని నోట్‌బుక్ లాంగ్ సంభాషణల్లో Microsoft Agent Framework ఉపయోగించి కాంటెక్స్ట్‌ను ఎలా నిర్వహించాలో చూపిస్తుంది. సంభాషణలు పెరుగుతున్నట్లుగానే, టోకెన్ సంఖ్య పెరుగుతుంది — మోడల్ కాంటెక్స్ట్ విండోని మించిపోతుంది. దీన్ని పరిష్కరించడానికి మేము **కాంటెక్స్ట్ సారాంశ నమూనా** మరియు **ఏజెంట్ స్క్రాచ్‌ప్యాడ్** ను ఉపయోగిస్తున్నాము, ఇది స్థిరమైన మెమొరీ అందిస్తుంది.

## మీరు నేర్చుకునేది:
1. **కాంటెక్స్ట్ నిర్వహణ ఎందుకు ముఖ్యం**: టోకెన్ పరిమితులు మరియు కాంటెక్స్ట్ విండోలను అర్ధం చేసుకోవడం
2. **కాంటెక్స్ట్-అవేర్ ఏజెంట్లు**: తమ సంభాషణ కాంటెక్స్ట్‌ను నిర్వహించే ఏజెంట్లను నిర్మించడం
3. **కాంటెక్స్ట్ సారాంశ నమూనా**: సంభాషణ చరిత్రను సంక్షిప్తం చేయడానికి సాధనాల ఉపయోగం
4. **ఏజెంట్ స్క్రాచ్‌ప్యాడ్**: కాంటెక్స్ట్ తగ్గింపును తట్టుకునే స్థిరమైన మెమొరీ

## అవశ్యకతలు:
- Azure OpenAI సెటప్, ఎన్‌విరాన్‌మెంట్ వేరియబుల్స్ సెట్ చేసినవి
- పూర్వ పాఠాల నుండి బేసిక్ ఏజెంట్ కాన్సెప్ట్‌ల అవగాహన


## సెటప్


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## సందర్భ నిర్వహణ ఎందుకు ముఖ్యం

ప్రతి LLM కి ఒక పరిమిత **సందర్భ విండో** ఉంటుంది — ఒకే అభ్యర్థనలో ఇది ప్రాసెస్ చేయగల టోకెన్ల గరిష్ట సంఖ్య. బహుళ-తిరుగు సంభాషణ కొనసాగుతుండగా:

- ప్రతి వినియోగదారు సందేశం మరియు సహాయకుడు జవాబు ఇచ్చే ప్రతిసారీ **టోకెన్ లెక్క రేఖీయంగా పెరుగుతుంది**.
- **ప్రాంప్ట్ టోకెన్లు ఖర్చును ప్రభావితం చేస్తాయి** ఎందుకంటే ప్రతి తిరుగులో పూర్తిగా చరిత్రను పునఃప్రేరేపిస్తారు.
- చివరికి సంభాషణ **సందర్భ విండోని మించిపోతుంది** మరియు నమూనా కట్ చేస్తుంది లేదా లోపాన్ని చూపిస్తుంది.

### సందర్భాన్ని నిర్వహించడానికీ వ్యూహాలు

| వ్యూహం | అది ఎలా పని చేస్తుంది | వ్యత్యాసం |
|---|---|---|
| **కట్ చేయడం** | పాత సందేశాలను తొలగించడం | మొదటి సందర్భం కోల్పోవడం |
| **సారాంశం** | పాత సందేశాలను ఒక సారాంశంగా సంగ్రహించడం | కొంత వివరాలు కోల్పోతారు, కానీ ముఖ్యాంశాలు నిలిచిపోతాయి |
| **Scratchpad / బయటి మెమరీ** | సంభాషణ వెలుపల ముఖ్యమైన విషయాలను నిల్వ చేయడం | టూల్ కాల్స్ అవసరం, కానీ ఏ తక్కువతనం ఎదురైనా నిలుస్తుంది |

ఈ నోట్‌బుక్‌లో మేము **సారాంశం** మరియు **scratchpad టూల్** ను కలిపి వాడుతున్నాము కాబట్టి ఏజెంట్ సంభాషణ చరిత్ర సంగ్రహించినప్పటికీ క kontinuty ని నిలుపుకోవచ్చు.


## కాంటెక్స్ట్-అవేర్ ఏజెంట్ సృష్టించడం


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## పొడవైన సంభాషణను అనుకరించడం  

చుట్టూ పరిసర పరిస్థితులు ఎలా సేకరించబడుతున్నాయో చూడటానికి మల్టీ-టర్న్ సంభాషణని మనం పరిశీలిద్దాం. ఏజెంట్ కీలక వివరాలను (ప్రముఖతలు, బడ్జెట్, ప్రయాణ తేదీలు) టర్న్లలో నిలుపుకుని, కొనసాగుదలని చూపించాలి.  


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

ఏజెంట్ మునుపటి టర్న్‌ల నుండి సాందర్భాన్ని ఎలా నిలబెట్టుకున్నానో గమనించండి — ఇది జపాన్, సుషి, ఆలయాలు, ఫోటోగ్రఫీ, $3000 బడ్జెట్, ఒకే వ్యక్తి ప్రయాణం, మరియు ఏప్రిల్ సమయం గురించి తెలుసుకొంటుంది. ఒక చిన్న సంభాషణలో ఇది బాగా పనిచేస్తుంది, కానీ సంభాషణ పెరిగేకొద్దీ పూర్తి చరిత్రను తిరిగి పంపించడం ఖరీదుగా మారుతుంది.

సాందర్భం సేకరణను చూసేందుకు మరిన్ని టర్న్‌లతో సంభాషణను కొనసాగించుదాం:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## సందర్భం సంగ్రహణ నమూనా

సంభాషణ పెరిగేకొద్ది, మనం సేకరించిన సందర్భాన్ని ఒక సంక్షిప్త ఆకారంలో చక్కబెట్టడానికి **సంగ్రహణ సాధనం** ఉపయోగించవచ్చు. ఏజెంట్ ఈ సాధనాన్ని పిలుచును, ముఖ్యమైన పరిగణనలు రికార్డ్ చేయడానికి, తద్వారా పాత సందేశాలు తొలగించినప్పటికీ, ముఖ్యమైన సమాచారం రక్షించబడుతుంది.

ఈ నమూనా అధిక శ్రేణి చరిత్ర తగ్గింపు కోసం పునాది కట్టదు:
1. ఏజెంట్ సంభాషణ నుండి ముఖ్యమైన విషయాలను గుర్తిస్తుంది
2. వాటిని నిల్వ చేయడానికి సంగ్రహణ సాధనాన్ని పిలుస్తుంది
3. పాత సందేశాలను సురక్షితంగా తొలగించవచ్చు, ఎందుకంటే సంగ్రహం ఏమి ముఖ్యమో cap్చర్లిస్తుంది

క్రింద మనం ఒక `summarize_preferences` సాధనాన్ని నిర్వచిస్తాము, దీన్ని ఏజెంట్ తన సేకరించిన సంగ్రహం కోసం పిలవవచ్చు.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## సారాంశం

ఈ పాఠంలో మీరు మైక్రోసాఫ్ట్ ఏజెంట్ ఫ్రేమ్‌వర్క్ ఉపయోగించి దీర్ఘకాలిక ఏజెంట్ సంభాషణలలో సందర్భాన్ని నిర్వహ하는 విధానం నేర్చుకున్నారు:

### ముఖ్యమైన భావనలు
- **సందర్భ విండోల పరిమితి ఉంది** — సంభాషణ చరిత్రలో ప్రతి టోకెన్ ఖర్చు అవుతుంది మరియు పరిమితికి చేర్చబడుతుంది.
- **సారాంశ సాధనాలు** ఏజెంట్ సేకరించిన సందర్భాన్ని సంక్షిప్త సారాంశాలుగా మార్చి, అవసరమైన సమాచారాన్ని నిలుపుకుంటూ టోకెన్ వాడకాన్ని తగ్గిస్తాయి.
- **ఏజెంట్ స్క్రాచ్‌ప్యాడ్లు** సంభాషణ తగ్గడం అయినప్పటికీ మన్నించే బాహ్య జ్ఞాపకాన్ని అందిస్తాయి.

### మీరు బిల్డ్ చేసినవి
- **సందర్భాన్ని తెలుసుకునే ఏజెంట్** దీర్ఘకాల సంభాషణలలో కొనసాగింపును కలిగించే
- **సారాంశ సాధనం** (`summarize_preferences`) ముఖ్యమైన వినియోగదారు వివరాలను సంక్షిప్త ఫార్మాట్‌లో నమోదు చేస్తుంది
- **బహుళ-తిరుగుడు సంభాషణ** సందర్భం నిలుపుకోవడం మరియు మార్పు నిర్వహణను చూపించే

### వాస్తవ ప్రపంచ ఉపయోగాలు
- **కస్టమర్ సర్వీస్ బాట్లు**: దీర్ఘకాల సహాయ సెషన్లలో ప్రాధాన్యతలను గుర్తుంచుకోవడం
- **వ్యక్తిగత సహాయకులు**: సందర్భం మళ్లీ చెప్పకుండా కొనసాగుతున్న ప్రాజెక్టులను ట్రాక్ చేయడం
- **శిక్షణా ఉపాధ్యాయులు**: అనేక సంభాషణలలో విద్యార్థి పురోగతిని నిలుపుకోవడం

### తర్వాతి దశలు
- ఫైల్ ఆధారిత స్థిరత్వంతో పూర్తి స్క్రాచ్‌ప్యాడ్ సాధనాన్ని అమలుచేయడం
- సారాంశం తర్వాత ఆటోమేటిక్ చరిత్ర సంక్షిప్తతను జోడించడం
- సాందర్భిక జ్ఞాపక శోధన కోసం వెక్టర్ డేటాబేస్‌లతో కలపడం
- పూర్తి సందర్భంతో రోజులు తర్వాత సంభాషణలను పునఃప్రారంభించగల ఏజెంట్లను తయారు చేయడం


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
